In [1]:
import math
import re
import numpy as np
import pandas as pd
import QuantLib as ql

Interp_term_structure = pd.read_excel("Interp_term_structure.xlsx")

In [2]:
#=============================================================================
# PART 0  –  Dates & conventions  
# =============================================================================

trade_date = ql.Date(5, 11, 2025)
calendar   = ql.TARGET()
spot_date  = calendar.advance(trade_date, 2, ql.Days)

bdc        = ql.ModifiedFollowing
dc_deposit = ql.Actual360()
dc_swap    = ql.Thirty360(ql.Thirty360.BondBasis)
dc_curve   = ql.Actual365Fixed()

# Python date equivalents (for output and q4 date arithmetic)
TRADE_DATE_PY = trade_date.to_date()
SPOT_DATE_PY  = spot_date.to_date()

print(f"Trade date : {trade_date}   Spot date : {spot_date}")

Trade date : November 5th, 2025   Spot date : November 7th, 2025


In [3]:

def days_from_spot(maturity):
    return maturity.serialNumber() - spot_date.serialNumber()

In [4]:
# =============================================================================
# Q4  Best- and Worst-Case Scenario Valuation
# =============================================================================

# ── Bond constants ────────────────────────────────────────────────────────────
N          = 1_000.0
ALPHA      = 0.25        # 30/360 quarterly accrual fraction (constant)
CAP_RATE   = 0.0545      # 5.45 % p.a.
FLOOR_RATE = 0.0000      # 0.00 % p.a.
PART       = 1.60        # participation factor
L_STAR     = CAP_RATE / PART   # 3.40625 %

# Period 6: fixed 10 Sep 2025, payable 12 Dec 2025
L6 = 0.02029
R6 = min(CAP_RATE, max(FLOOR_RATE, PART * L6))   # 0.032464
C6 = N * R6 * ALPHA                               # 8.116 EUR

# ── Payment schedule periods 6-40  (from KID, Following BDC already applied) ─
# Using QuantLib dates so we can call days_from_spot() directly
def qldate(d, m, y):
    return ql.Date(d, m, y)

SCHEDULE_QL = {
     6: qldate(12, 12, 2025),
     7: qldate(12,  3, 2026),
     8: qldate(12,  6, 2026),
     9: qldate(14,  9, 2026),
    10: qldate(14, 12, 2026),
    11: qldate(12,  3, 2027),
    12: qldate(14,  6, 2027),
    13: qldate(13,  9, 2027),
    14: qldate(13, 12, 2027),
    15: qldate(13,  3, 2028),
    16: qldate(12,  6, 2028),
    17: qldate(12,  9, 2028),
    18: qldate(12, 12, 2028),
    19: qldate(12,  3, 2029),
    20: qldate(12,  6, 2029),
    21: qldate(12,  9, 2029),
    22: qldate(12, 12, 2029),
    23: qldate(12,  3, 2030),
    24: qldate(12,  6, 2030),
    25: qldate(12,  9, 2030),
    26: qldate(12, 12, 2030),
    27: qldate(12,  3, 2031),
    28: qldate(12,  6, 2031),
    29: qldate(12,  9, 2031),
    30: qldate(12, 12, 2031),
    31: qldate(12,  3, 2032),
    32: qldate(14,  6, 2032),
    33: qldate(13,  9, 2032),
    34: qldate(13, 12, 2032),
    35: qldate(14,  3, 2033),
    36: qldate(13,  6, 2033),
    37: qldate(12,  9, 2033),
    38: qldate(12, 12, 2033),
    39: qldate(13,  3, 2034),
    40: qldate(12,  6, 2034),   # maturity
}
MATURITY_QL = SCHEDULE_QL[40]

# ── Discount factor lookup from Interp_term_structure ────────────────────────
_days_arr = Interp_term_structure["Days"].to_numpy(dtype=float)
_df_arr   = Interp_term_structure["DF"].to_numpy(dtype=float)

def get_df(ql_date: ql.Date) -> float:
    """
    Discount factor P(spot, d) by linear interpolation on the
    day grid from Interp_term_structure.
    Days measured from spot_date using QuantLib serial numbers,
    matching exactly how days_from_spot() works in q3.
    """
    d = ql_date.serialNumber() - spot_date.serialNumber()
    if d <= 0:
        return 1.0
    return float(np.interp(d, _days_arr, _df_arr))

# ── Accrued interest  (30/360 day count) ─────────────────────────────────────
# Period 6 runs 12 Sep 2025 -> 12 Dec 2025.  Trade date = 5 Nov 2025.
# 30/360 formula: days = (Y2-Y1)*360 + (M2-M1)*30 + (D2-D1)
#                      = 0*360 + (11-9)*30 + (5-12) = 53 days
_days_acc = (2025 - 2025) * 360 + (11 - 9) * 30 + (5 - 12)   # 53
AI        = N * R6 * (_days_acc / 360)                         # 4.779 EUR

print(f"\nAccrued Interest")
print(f"  Period 6 coupon rate  R6 : {R6*100:.4f}%")
print(f"  Days accrued (30/360)    : {_days_acc}")
print(f"  AI = 1000 x {R6*100:.4f}% x {_days_acc}/360 = EUR {AI:.3f}")

# ── Worst-case valuation ──────────────────────────────────────────────────────
def value_worst_case() -> dict:
    """
    EURIBOR <= 0 at all future resets -> floor binds -> coupons 7-40 = 0
    V_min = C6 * P(spot, T6) + N * P(spot, T40)
    """
    df6  = get_df(SCHEDULE_QL[6])
    df40 = get_df(MATURITY_QL)
    pv_c6  = C6 * df6
    pv_par = N  * df40
    gross  = pv_c6 + pv_par
    clean  = gross - AI
    return dict(
        days_6    = days_from_spot(SCHEDULE_QL[6]),
        days_40   = days_from_spot(MATURITY_QL),
        df6       = round(df6,   6),
        df40      = round(df40,  6),
        pv_c6     = round(pv_c6, 3),
        pv_par    = round(pv_par,3),
        gross     = round(gross, 2),
        gross_pct = round(gross / N * 100, 2),
        ai        = round(AI,    3),
        clean     = round(clean, 2),
        clean_pct = round(clean / N * 100, 2),
    )

# ── Best-case valuation ───────────────────────────────────────────────────────
def value_best_case() -> tuple:
    """
    EURIBOR >= L* at all future resets -> cap binds -> coupons 7-40 = EUR 13.625
    V_max = C6*P(spot,T6) + C_max * sum_{i=7}^{40} P(spot,Ti) + N*P(spot,T40)
    Returns (summary dict, detail DataFrame)
    """
    C_MAX = N * CAP_RATE * ALPHA   # 13.625 EUR

    df6   = get_df(SCHEDULE_QL[6])
    pv_c6 = C6 * df6

    rows    = []
    annuity = 0.0
    for period, ql_date in SCHEDULE_QL.items():
        if period < 7:
            continue
        d_i  = days_from_spot(ql_date)
        df_i = get_df(ql_date)
        pv_i = C_MAX * df_i
        annuity += df_i
        rows.append(dict(
            Period       = period,
            Payment_Date = ql_date.ISO(),
            Days         = d_i,
            DF           = round(df_i, 6),
            Coupon_EUR   = round(C_MAX, 3),
            PV_EUR       = round(pv_i,  4),
        ))

    df40   = get_df(MATURITY_QL)
    pv_ann = C_MAX * annuity
    pv_par = N * df40
    gross  = pv_c6 + pv_ann + pv_par
    clean  = gross - AI

    summary = dict(
        df6         = round(df6,     6),
        pv_c6       = round(pv_c6,   3),
        annuity_sum = round(annuity, 4),
        pv_annuity  = round(pv_ann,  3),
        df40        = round(df40,    6),
        pv_par      = round(pv_par,  3),
        gross       = round(gross,   2),
        gross_pct   = round(gross / N * 100, 2),
        ai          = round(AI,      3),
        clean       = round(clean,   2),
        clean_pct   = round(clean / N * 100, 2),
    )
    return summary, pd.DataFrame(rows)

# ── Run ───────────────────────────────────────────────────────────────────────
wc            = value_worst_case()
bc, bc_detail = value_best_case()
C_MAX         = N * CAP_RATE * ALPHA

# ── Print results ─────────────────────────────────────────────────────────────
print("\n" + "=" * 66)
print("Q4 -- Best- and Worst-Case Scenario Valuation")
print(f"   Trade date : {trade_date}   |   Spot date : {spot_date}")
print("=" * 66)

print("\n" + "-" * 66)
print("WORST CASE  --  EURIBOR <= 0% at all future resets")
print("-" * 66)
print(f"  Coupons 7-40  :  EUR 0.000  (0% floor binds)")
print(f"  Period-6 PV   :  EUR {C6:.3f} x DF({wc['df6']:.6f}) = EUR {wc['pv_c6']:.3f}")
print(f"  Principal PV  :  EUR {N:.0f}  x DF({wc['df40']:.6f}) = EUR {wc['pv_par']:.3f}")
print(f"\n  Gross price   :  EUR {wc['gross']:.2f}  ({wc['gross_pct']:.2f}% of par)")
print(f"  Accrued int.  :  EUR {wc['ai']:.3f}")
print(f"  Clean price   :  EUR {wc['clean']:.2f}  ({wc['clean_pct']:.2f}% of par)")

print("\n" + "-" * 66)
print(f"BEST CASE  --  EURIBOR >= L* = {L_STAR*100:.5f}% at all future resets")
print("-" * 66)
print(f"  Max coupon / period  :  EUR {C_MAX:.3f}")
print(f"  Period-6 PV          :  EUR {C6:.3f} x DF({bc['df6']:.6f}) = EUR {bc['pv_c6']:.3f}")
print(f"  Annuity sum DF(7-40) :  {bc['annuity_sum']:.4f}")
print(f"  Capped annuity PV    :  EUR {C_MAX:.3f} x {bc['annuity_sum']:.4f} = EUR {bc['pv_annuity']:.3f}")
print(f"  Principal PV         :  EUR {N:.0f} x DF({bc['df40']:.6f}) = EUR {bc['pv_par']:.3f}")
print(f"\n  Gross price          :  EUR {bc['gross']:.2f}  ({bc['gross_pct']:.2f}% of par)")
print(f"  Accrued int.         :  EUR {bc['ai']:.3f}")
print(f"  Clean price          :  EUR {bc['clean']:.2f}  ({bc['clean_pct']:.2f}% of par)")

rng_gross = bc["gross"] - wc["gross"]
rng_clean = bc["clean"] - wc["clean"]
print("\n" + "=" * 66)
print("SUMMARY")
print("=" * 66)
print(f"{'Scenario':<30} {'Gross EUR':>10} {'Gross %':>9} {'Clean %':>9}")
print("-" * 62)
print(f"{'Worst case (floor binds)':<30} {wc['gross']:>10.2f} {wc['gross_pct']:>9.2f} {wc['clean_pct']:>9.2f}")
print(f"{'Best case (cap binds)':<30} {bc['gross']:>10.2f} {bc['gross_pct']:>9.2f} {bc['clean_pct']:>9.2f}")
print("-" * 62)
print(f"{'Range (best - worst)':<30} {rng_gross:>10.2f} {rng_gross/10:>9.2f} {rng_clean/10:>9.2f}")
print("\nNote: neither scenario is the fair value.")
print("  Worst case ignores option value of the 0% floor.")
print("  Best case  ignores cost of the embedded short caplet strip.")
print("  The true risk-neutral value lies strictly between these bounds.")

print("\nBest-case per-period detail (periods 7-40):")
print(bc_detail.to_string(index=False))

# ── Save outputs ──────────────────────────────────────────────────────────────
summary_df = pd.DataFrame([
    {
        "Scenario"  : "Worst Case",
        "Condition" : "EURIBOR <= 0 (floor binds, coupons 7-40 = 0)",
        "Gross EUR" : wc["gross"],
        "Gross %"   : wc["gross_pct"],
        "AI EUR"    : wc["ai"],
        "Clean %"   : wc["clean_pct"],
    },
    {
        "Scenario"  : "Best Case",
        "Condition" : "EURIBOR >= 3.406% (cap binds, coupons 7-40 = 13.625)",
        "Gross EUR" : bc["gross"],
        "Gross %"   : bc["gross_pct"],
        "AI EUR"    : bc["ai"],
        "Clean %"   : bc["clean_pct"],
    },
])

summary_df.to_csv("q4_scenarios.csv",        index=False)
bc_detail.to_csv( "q4_best_case_detail.csv",  index=False)

print("Saved: q4_scenarios.csv")
print("Saved: q4_best_case_detail.csv")


Accrued Interest
  Period 6 coupon rate  R6 : 3.2464%
  Days accrued (30/360)    : 53
  AI = 1000 x 3.2464% x 53/360 = EUR 4.779

Q4 -- Best- and Worst-Case Scenario Valuation
   Trade date : November 5th, 2025   |   Spot date : November 7th, 2025

------------------------------------------------------------------
WORST CASE  --  EURIBOR <= 0% at all future resets
------------------------------------------------------------------
  Coupons 7-40  :  EUR 0.000  (0% floor binds)
  Period-6 PV   :  EUR 8.116 x DF(0.998136) = EUR 8.101
  Principal PV  :  EUR 1000  x DF(0.802378) = EUR 802.378

  Gross price   :  EUR 810.48  (81.05% of par)
  Accrued int.  :  EUR 4.779
  Clean price   :  EUR 805.70  (80.57% of par)

------------------------------------------------------------------
BEST CASE  --  EURIBOR >= L* = 3.40625% at all future resets
------------------------------------------------------------------
  Max coupon / period  :  EUR 13.625
  Period-6 PV          :  EUR 8.116 x DF(0.9981